In [0]:
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("load_type", "FULL")
dbutils.widgets.text("merge_keys", "")

table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
load_type = dbutils.widgets.get("load_type")
merge_keys = dbutils.widgets.get("merge_keys")

In [0]:
bronze_table=spark.read.table(f"caresyncdata.bronze.{table_name}").filter(f"landing_timestamp='{landing_timestamp}'")

In [0]:
from pyspark.sql.functions import *
df_update=bronze_table.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp",current_timestamp())



In [0]:
target_table = f"caresyncdata.silver.{table_name}"
load_type_upper = load_type.upper().strip()

if load_type_upper == "FULL":
    df_update.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)

elif load_type_upper == "APPEND":
    df_update.write.mode("append").saveAsTable(target_table)

elif load_type_upper == "MERGE":
    if not merge_keys.strip():
        raise ValueError("merge_keys widget must be set for MERGE load type (comma-separated column names)")

    if not spark.catalog.tableExists(target_table):
        # Target table doesn't exist yet — create it with an initial full load
        print(f"Target table '{target_table}' does not exist. Performing initial full load...")
        df_update.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
    else:
        keys = [k.strip() for k in merge_keys.split(",")]
        df_update.createOrReplaceTempView("_merge_source")
        merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])
        spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING _merge_source AS source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Valid values: FULL, APPEND, MERGE")

print(f"[{load_type_upper}] Write to '{target_table}' completed successfully.")

In [0]:
records_count=df_update.count()
dbutils.notebook.exit(str(records_count))